In [ ]:
# Un/binding events
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt

from mg_sampling import PROJECT_DIR
from mg_sampling.tools import edges_to_range, read_plumed
from mg_sampling.plot import LABELS

In [ ]:
def plot_unique_event_rate(df):
    inner = df.query('n_inner > 0 & name == "MG"')
    labels = inner['index'].astype(str) + inner['inner_indices']
    _, unique_indices = np.unique(labels, return_index=True)
    unique_times = inner.iloc[unique_indices]['time'] / 1000  # ns
    τ = 20 # Sliding window (ns)
    dt = 0.1
    counts, t_edges = np.histogram(unique_times, bins=np.arange(0, unique_times.max()+dt, dt))
    t_range = edges_to_range(t_edges)
    rate = counts / dt
    n_window = int(τ//dt)
    window = np.hanning(n_window)
    rate = np.convolve(rate, window / window.sum(), mode='valid')
    fig, ax = plt.subplots()
    ax.plot(t_range[n_window//2:n_window//2+len(rate)], rate, color='black')
    ax.set_ylabel('New binding events [ns⁻¹]')
    ax.set_xlabel('Time [ns]')
    ax2 = ax.twinx()
    ax2.scatter(unique_times, np.random.normal(size=len(unique_times)), s=0.1, color='gray', rasterized=True)
    ax2.set_yticks([])
    ax.set_ylim(0, None)
    return ax

systems = ['7ez0', '7ez0_noion', '7ez0_2', '7ez0_noion_2']
forcefields = ['mMg_opc', 'mMg_tip3p', 'nMg_tip3p', 'ecc_spce']
repids = [0]

for forcefield in forcefields:
    for system in systems:
        for repid in repids:
            path = PROJECT_DIR / 'replex' / system / forcefield / f'rep{repid:02d}'
            df = pd.read_csv(path / 'ion_analysis.csv')
            opt_times = np.atleast_1d(np.loadtxt(path / '../opt_times.txt')) / 1000
            ax = plot_unique_event_rate(df)
            l = None
            for t in opt_times:
                l = ax.axvline(t, color='royalblue', ls='dashed')
            if l is not None:
                ax.legend([l], ['Optimize scalings'])
            plt.savefig(f'figures/replex/unique_event_rate_{system}_{forcefield}.pdf')
            plt.title(f"Repl. Ex. — {LABELS[system]} {LABELS[forcefield]}")


In [ ]:
system = '7ez0_noion'
forcefield = 'nMg_tip3p'
for repid in range(4):
    path = PROJECT_DIR / 'plain_md' / system / forcefield / f'rep{repid:02d}'
    df = pd.read_csv(path / 'ion_analysis.csv')
    ax = plot_unique_event_rate(df)
    plt.savefig(f'figures/plain_md/unique_event_rate_{system}_{forcefield}_{repid:02d}.pdf')
    plt.title(f"Conventional MD — {LABELS[system]} {LABELS[forcefield]}")

In [ ]:
from bussilab.wham import wham
import glob
from mg_sampling.constants import RT
from scipy.special import logsumexp
system = '7ez0'
forcefield = 'mMg_opc'
repids = range(12)
path = PROJECT_DIR / 'replex' / system / forcefield

scalings = [np.loadtxt(f) for f in glob.glob(str(path / 'scalings.??.txt'))]
opt_times = np.insert(np.loadtxt(path / 'opt_times.txt'), 0, 0.0)

bias = []
for repid in repids:
    cv = read_plumed(path / f'rep{repid:02d}' / f'bias.COLVAR.{repid}')
    _bias = []
    for i in range(len(scalings)-1):
        mask = (opt_times[i] <= cv['time']) * (opt_times[i+1] > cv['time'])
        _bias.append(np.asarray(cv[mask]['cv'])[:, None] * scalings[i])
    mask = (opt_times[-1] <= cv['time'])
    _bias.append(np.asarray(cv[mask]['cv'])[:, None] * scalings[-1])
    bias.append(np.concatenate(_bias))
bias = np.concatenate(bias) / RT

In [ ]:
W = wham(bias)
λ_range = np.linspace(0, 1, 20)
dλ = λ_range[1] - λ_range[0]

In [ ]:
# logw that sample x comes from λ
import matplotlib as mpl
logW = W.logW[:, None] - λ_range * bias[:, -1][:, None]  # p(x, λ)
logWλ = logsumexp(logW, axis=0)  # p(λ)
logW -= logWλ  # p(x|λ)
logWx = logsumexp(logW, axis=1)

logWλλ = logsumexp(
    logW[:, :, None] + logW[:, None, :] - logWx[:, None, None],
    axis=0,
)
logWλλ -= logsumexp(logWλλ, axis=0) + np.log(dλ)
for i, λ in enumerate(λ_range):
    plt.plot(λ_range, np.exp(logWλλ[i]), color=plt.cm.magma(λ))
plt.colorbar(
    mpl.cm.ScalarMappable(
        norm=mpl.colors.Normalize(0, 1),
        cmap=mpl.cm.magma,
    ),
    ax=plt.gca(),
    label=r"$λ'$",
)
plt.xlabel(r"$λ$")
plt.ylabel(r"$ρ(λ|λ')$")

In [ ]:
# logw that sample x comes from λ
import matplotlib as mpl
logW = W.logW[:, None] - scalings[0] * bias[:, -1][:, None]  # p(x, λ)
logWλ = logsumexp(logW, axis=0)  # p(λ)
logW -= logWλ  # p(x|λ)
logWx = logsumexp(logW, axis=1)

logWλλ = logsumexp(
    logW[:, :, None] + logW[:, None, :] - logWx[:, None, None],
    axis=0,
)
logWλλ -= logsumexp(logWλλ, axis=0) + np.log(dλ)
for i, λ in enumerate(scalings[0]):
    plt.plot(λ_range, np.exp(logWλλ[i]), color=plt.cm.magma(λ))
plt.colorbar(
    mpl.cm.ScalarMappable(
        norm=mpl.colors.Normalize(0, 1),
        cmap=mpl.cm.magma,
    ),
    ax=plt.gca(),
    label=r"$λ'$",
)
plt.xlabel(r"$λ$")
plt.ylabel(r"$ρ(λ|λ')$")

In [ ]:
replica_temp = np.loadtxt(path / 'replica_temp.xvg')

In [ ]:
# Probability of being in j after N steps starting from i
rex_τ = 1  # ps
Ns = [10, 100, 1000, 10000, 100000]
fig, axarr = plt.subplots(len(Ns), figsize=(8, 3*len(Ns)), sharex=True, sharey=True)
for ax, N in zip(axarr.flatten(), Ns):
    reprange = np.arange(12)
    js = []
    for i, temps in enumerate(replica_temp[:-N, 1:]):
        js.append(replica_temp[i+N, 1:][np.argsort(temps)])
    js = np.array(js)
    counts = (js[:, :, None] == reprange).mean(axis=0)

    for i in reprange:
        ax.plot(scalings[0], counts[i], color=plt.cm.magma(scalings[0][i]), marker='.')
    plt.colorbar(
        mpl.cm.ScalarMappable(
            norm=mpl.colors.Normalize(0, 1),
            cmap=mpl.cm.magma,
        ),
        ax=ax,
        label=r"$λ'$",
    )
    ax.set_xlabel(r"$λ$")
    ax.set_ylabel(r"$\langle P(λ,t+τ|λ',t)\rangle$")
    ax.set_title(f"τ = {rex_τ * N} ps")
plt.tight_layout()